# Webcam face recognition

Three independent examples compare `face_recognition`, DeepFace with ArcFace, and OpenCV's LBPH recognizer. Run one code cell at a time in an environment with a webcam and a graphical display; press **q** in the camera window to stop it. These examples require separate libraries: `face_recognition`, `deepface`, and `opencv-contrib-python`, respectively. DeepFace may download model weights on its first run.

The first example reads one clear reference face per image from `known_faces/`. The other two read images from `faces_db/<person>/`. These personal reference images are not included. Recognition thresholds are illustrative and should be evaluated on your own data; the LBPH example expects cropped face images in its database.

These interactive webcam examples were not executed during the notebook cleanup.

**Data:** No fixed dataset. The examples use a webcam and locally supplied reference images; those files are not included.

In [ ]:
import face_recognition
import cv2
import os
from pathlib import Path

known_encodings = []
known_names = []

known_dir = Path("known_faces")
if not known_dir.is_dir():
    raise FileNotFoundError("Create known_faces/ and add one reference image per person.")
for file in sorted(known_dir.iterdir()):
    if file.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue
    img = face_recognition.load_image_file(str(file))
    encodings = face_recognition.face_encodings(img)
    if len(encodings) != 1:
        print(f"Skipping {file.name}: expected exactly one face, found {len(encodings)}")
        continue
    known_encodings.append(encodings[0])
    known_names.append(file.stem)
if not known_encodings:
    raise ValueError("No usable reference faces found in known_faces/.")

video = cv2.VideoCapture(0)

while True:
    ret, frame = video.read()

    if not ret:
        print("Could not read a frame from the camera.")
        break
    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    locations = face_recognition.face_locations(rgb)
    encodings = face_recognition.face_encodings(rgb, locations)

    for face_encoding, (top, right, bottom, left) in zip(encodings, locations):
        distances = face_recognition.face_distance(known_encodings, face_encoding)
        index = distances.argmin()
        name = known_names[index] if distances[index] <= 0.6 else "Unknown"

        cv2.rectangle(frame, (left, top), (right, bottom), (0,255,0), 2)
        cv2.putText(frame, name, (left, top - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

from deepface import DeepFace
import cv2
import uuid

DB_PATH = "faces_db"
if not os.path.isdir(DB_PATH):
    raise FileNotFoundError("Create faces_db/ with a subfolder for each person.")

video = cv2.VideoCapture(0)

while True:
    ret, frame = video.read()
    if not ret:
        print("Could not read a frame from the camera.")
        break
    frame = cv2.flip(frame, 1)

    temp_path = f"/tmp/{uuid.uuid4()}.jpg"
    cv2.imwrite(temp_path, frame)  # нужно для DeepFace.find()

    try:
        result = DeepFace.find(
            img_path=temp_path,
            db_path=DB_PATH,
            model_name="ArcFace",
            detector_backend="mediapipe",
        )

        if len(result) > 0 and len(result[0]) > 0:
            identity = result[0].iloc[0]["identity"]
            name = os.path.basename(os.path.dirname(identity))
        else:
            name = "Unknown"

    except Exception as e:
        print("Ошибка:", e)
        name = "No face detected"

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)  # удаляем временный файл

    cv2.putText(frame, name, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("DeepFace ArcFace webcam", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

video.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import os
import numpy as np

recognizer = cv2.face.LBPHFaceRecognizer_create()

faces = []
labels = []
names = {}
label_id = 0

# load training images
for name in os.listdir("faces_db"):
    folder = os.path.join("faces_db", name)
    if not os.path.isdir(folder):
        continue
    names[label_id] = name

    for file in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, file), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            faces.append(img)
            labels.append(label_id)

    label_id += 1

if not faces:
    raise ValueError("No readable reference images found in faces_db/.")
recognizer.train(faces, np.array(labels, dtype=np.int32))

# Load the detector once, outside the frame loop.
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
if face_cascade.empty():
    raise RuntimeError("Could not load the OpenCV face detector.")

# webcam stream
video = cv2.VideoCapture(0)

while True:
    ret, frame = video.read()
    if not ret:
        print("Could not read a frame from the camera.")
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x,y,w,h) in faces:
        roi = gray[y:y+h, x:x+w]
        label, confidence = recognizer.predict(roi)

        name = names[label] if confidence < 80 else "Unknown"

        cv2.rectangle(frame, (x,y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, name, (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("OpenCV LBPH", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()
